# AQR "Hold the Dip" Replication

This notebook is a simplified, SPY-based replication of the buy-the-dip analysis discussed in AQR's "Hold the Dip." The goal is not to exactly reproduce AQR's historical dataset or published numbers. Instead, this notebook tests whether drawdown-triggered dip-buying rules produce attractive results when applied to SPY from 1993 onward.

The analysis compares buy-and-hold against a grid of dip-buying strategies using CAGR, annualized volatility, Sharpe ratio, max drawdown, beta, alpha, correlation, and R-squared.

## Methodology

The strategy definition is intentionally simple:

1. Measure the current drawdown from the highest SPY close over a rolling lookback window.
2. Trigger a buy signal when the drawdown is at least the chosen threshold.
3. Stay invested for a fixed holding period after the signal.
4. Shift positions by one trading day to avoid lookahead bias.
5. Assume uninvested capital earns 0% and ignore transaction costs in this version.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

## Load SPY Data

In [ ]:
spy = yf.download("SPY", start="1993-01-01", auto_adjust=True, progress=False)

prices = spy["Close"].squeeze()
daily_returns = prices.pct_change().fillna(0)

spy.tail()

## Buy-and-Hold Baseline

In [ ]:
buy_hold_equity = (1 + daily_returns).cumprod()
buy_hold_drawdown = buy_hold_equity / buy_hold_equity.cummax() - 1

price_drawdown = prices / prices.cummax() - 1

print(f"Sample start: {prices.index.min().date()}")
print(f"Sample end:   {prices.index.max().date()}")
print(f"Max price drawdown: {price_drawdown.min():.2%}")

In [ ]:
def compute_stats(returns, equity_curve):
    returns = returns.squeeze().fillna(0)
    equity_curve = equity_curve.squeeze()

    years = len(returns) / 252
    cagr = equity_curve.iloc[-1] ** (1 / years) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe = np.nan if returns.std() == 0 else returns.mean() / returns.std() * np.sqrt(252)
    drawdown = equity_curve / equity_curve.cummax() - 1

    return pd.Series({
        "CAGR": cagr,
        "Annualized Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": drawdown.min(),
    })


buy_hold_stats = compute_stats(daily_returns, buy_hold_equity)
buy_hold_stats

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

buy_hold_equity.plot(ax=axes[0], logy=True, label="Buy and Hold")
axes[0].set_title("SPY Buy-and-Hold Growth of $1")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()

buy_hold_drawdown.plot(ax=axes[1], color="tab:red", label="Drawdown")
axes[1].set_title("Buy-and-Hold Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].legend()

plt.tight_layout()

## Drawdown-Triggered Buy-the-Dip Strategy

In [ ]:
def run_btd_strategy(prices, daily_returns, depth, lookback=21, hold=21):
    prices = prices.squeeze()
    daily_returns = daily_returns.squeeze().fillna(0)

    rolling_peak = prices.rolling(lookback).max()
    rolling_drawdown = prices / rolling_peak - 1

    buy_signal = rolling_drawdown <= depth
    position = buy_signal.rolling(hold).max().fillna(0)
    position = position.shift(1).fillna(0)

    strategy_returns = position * daily_returns
    equity_curve = (1 + strategy_returns).cumprod()

    return strategy_returns, equity_curve, position

In [ ]:
example_returns, example_equity, example_position = run_btd_strategy(
    prices=prices,
    daily_returns=daily_returns,
    depth=-0.05,
    lookback=21,
    hold=21,
)

example_stats = compute_stats(example_returns, example_equity)

pd.DataFrame({
    "Buy and Hold": buy_hold_stats,
    "BTD: 5% depth, 1M lookback, 1M hold": example_stats,
})

In [ ]:
comparison = pd.DataFrame({
    "Buy and Hold": buy_hold_equity,
    "BTD: 5% depth, 1M lookback, 1M hold": example_equity,
})

comparison.plot(figsize=(12, 6), logy=True)
plt.title("Buy-and-Hold vs Simple Buy-the-Dip Rule")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## Parameter Grid

In [ ]:
depths = [-0.05, -0.10, -0.15, -0.20]

lookbacks = {
    "1W": 5,
    "2W": 10,
    "3W": 15,
    "1M": 21,
    "3M": 63,
    "6M": 126,
    "1Y": 252,
}

holds = {
    "1M": 21,
    "2M": 42,
    "3M": 63,
    "6M": 126,
    "1Y": 252,
    "3Y": 756,
    "5Y": 1260,
}

all_returns = {}
all_equity = {}
all_positions = {}
rows = []

for depth in depths:
    for lookback_name, lookback in lookbacks.items():
        for hold_name, hold in holds.items():
            strategy_name = f"{int(abs(depth) * 100)}% depth, {lookback_name} lookback, {hold_name} hold"

            returns, equity, position = run_btd_strategy(
                prices=prices,
                daily_returns=daily_returns,
                depth=depth,
                lookback=lookback,
                hold=hold,
            )

            all_returns[strategy_name] = returns
            all_equity[strategy_name] = equity
            all_positions[strategy_name] = position

            stats = compute_stats(returns, equity)
            stats["Depth"] = depth
            stats["Lookback"] = lookback_name
            stats["Hold"] = hold_name
            stats["Strategy"] = strategy_name
            rows.append(stats)

results = pd.DataFrame(rows)
returns_df = pd.DataFrame(all_returns)
equity_df = pd.DataFrame(all_equity)
positions_df = pd.DataFrame(all_positions)

results.shape

In [ ]:
results.sort_values("CAGR", ascending=False).head(10)

In [ ]:
best_idx = results["CAGR"].idxmax()
best_strategy = results.loc[best_idx, "Strategy"]

best_btd_stats = results.loc[
    best_idx,
    ["CAGR", "Annualized Volatility", "Sharpe Ratio", "Max Drawdown"],
]

pd.DataFrame({
    "Buy and Hold": buy_hold_stats,
    "Best BTD by CAGR": best_btd_stats,
})

In [ ]:
best_comparison = pd.DataFrame({
    "Buy and Hold": buy_hold_equity,
    best_strategy: equity_df[best_strategy],
})

best_comparison.plot(figsize=(12, 6), logy=True)
plt.title("Buy-and-Hold vs Best Buy-the-Dip Strategy by CAGR")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## Market Exposure and Regression Diagnostics

In [ ]:
regression_rows = []

for strategy in returns_df.columns:
    data = pd.DataFrame({
        "Strategy": returns_df[strategy],
        "SPY": daily_returns,
    }).dropna()

    beta = data["Strategy"].cov(data["SPY"]) / data["SPY"].var()
    daily_alpha = data["Strategy"].mean() - beta * data["SPY"].mean()
    annual_alpha = 252 * daily_alpha
    correlation = data["Strategy"].corr(data["SPY"])
    r_squared = correlation ** 2
    exposure = positions_df[strategy].mean()

    regression_rows.append({
        "Strategy": strategy,
        "Beta": beta,
        "Annual Alpha": annual_alpha,
        "Correlation": correlation,
        "R Squared": r_squared,
        "Average Exposure": exposure,
    })

regression_results = pd.DataFrame(regression_rows)
full_results = results.merge(regression_results, on="Strategy")

full_results.sort_values("Annual Alpha", ascending=False).head(10)

In [ ]:
full_results.loc[
    full_results["Strategy"].eq(best_strategy),
    [
        "Strategy",
        "CAGR",
        "Sharpe Ratio",
        "Max Drawdown",
        "Beta",
        "Annual Alpha",
        "Correlation",
        "R Squared",
        "Average Exposure",
    ],
]

## Discrepancies in Results Between this Notebook and AQR's Results

Many instances in my methodology may have resulted in different data prior to analysis; let's explore these potential factors.

1. The data used was different. I used SPY from 1993 onward, while AQR used a broader set of historical market data, meaning that their results likely had additional merit.

2. I was unsure how they calculated rolling drawdown, so I implemented a sliding window approach. I thought that this was a good idea, because a long, slow decline is not the same as a "dip" in my opinion. AQR may have done something different, or may have a completely different interpretation.

3. Slight methodology differences may also make a difference. I mentioned my rolling window approach, but other ideas like assuming cash return while money is uninvested, which I did **not** do, can result in large differences.

Overall, these differences provide several plausible explanations for why my results differ from AQR's. In particular, the shorter SPY sample, simplified implementation assumptions, and alternative dip definition all affect strategy behavior. Additionally, AQR evaluates buy-the-dip strategies over a broader market history and across many market environments, whereas this notebook focuses exclusively on SPY. Consequently, this project should be viewed as a faithful conceptual replication rather than an exact numerical reproduction.

Despite these differences, the primary objective of this project was not to reproduce AQR's exact numerical results, but to replicate the underlying methodology and evaluate whether similar conclusions emerge under a simplified SPY-based implementation.

I will **not** be buying (nor holding) the dip, no matter what polymarket chatters seem to believe.

Next steps include adding transaction costs, modeling cash returns while out of the market, and comparing these dip-buying rules against trend-following benchmarks such as a 10-month moving average and 12-month time-series momentum.